# TrendPulse – Part 1: Python Foundations

## Project Objective
Analyze YouTube trending videos across the US, IN and GB.

## Dataset
Trending YouTube Video Statistics.

## Countries
US, GB and IN.

## Analysis Approach
Python fundamentals, data cleaning, data structures,
loops, functions and rule-based classification.



In [39]:
#required libraries
import pandas as pd
from datetime import datetime, date
from pathlib import Path
#project configuration
#project name
PROJECT_NAME = "TrendPulse"
#file path
DATA_DIR = Path("data")
#countries
COUNTRIES = ["US", "GB", "IN"]
#threshold
MINIMUM_RECORDS = 40000

VIRAL_VIEW_THRESHOLD = 1000000
STEADY_VIEW_THRESHOLD = 100000
HIGH_LIKE_RATIO = 0.80
DIVISIVE_LIKE_RATIO = 0.40
#execution time
run_started = datetime.now()
#print project details
print(f"PROJECT NAME : {PROJECT_NAME}")
print(f"COUNTRIES    : {', '.join(COUNTRIES)}")
print(f"STARTED AT   : {run_started.strftime('%Y-%m-%d %H:%M:%S')}")


PROJECT NAME : TrendPulse
COUNTRIES    : US, GB, IN
STARTED AT   : 2026-11-07 12:13:51



## Loading functions

Load the YouTube trending
video datasets from three countries using a reusable
Python function.

The loaded records will be converted into a list
of dictionaries for further processing.

   Read one country CSV and return rows as a list of dictionaries.

path      — file path as a string (e.g. "data/USvideos.csv")

*country*   — country code to add on every row (e.g. "US")

max_rows  — optional limit (default None = load all rows)
    


In [40]:
REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled", "video_error_or_removed",
]


def load_country(path, country, max_rows=None):

    df = pd.read_csv(path, nrows=max_rows)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")

    df["country"] = country
    rows = df.to_dict("records")
    print(f"Loaded {len(rows):,} records from {country} ({path})")
    return rows

records = []
for code in COUNTRIES:
    file_path = DATA_DIR / f"{code}videos.csv"
    country_rows = load_country(file_path, code)
    records.extend(country_rows)

print(f"Total records loaded: {len(records):,}")

cat_df = pd.read_csv(DATA_DIR / "categories.csv")
category_names = {}
for row in cat_df.to_dict("records"):
    category_names[int(row["category_id"])] = row["category_name"]

print(f"Category lookup size: {len(category_names)}")
print(f"Example: 10 -> {category_names.get(10)}")

Loaded 16,080 records from US (data/USvideos.csv)
Loaded 15,075 records from GB (data/GBvideos.csv)
Loaded 15,075 records from IN (data/INvideos.csv)
Total records loaded: 46,230
Category lookup size: 17
Example: 10 -> Music


## Type Conversion and Cleaning
Try to turn a value into an int. On failure, count it and return 0.

Try to turn a value into True/False. On failure, count it and return False.

Parse yy.dd.mm into a date object.




In [41]:
failed = {
    "views": 0,
    "likes": 0,
    "dislikes": 0,
    "comment_count": 0,
    "trending_date": 0,
    "publish_time": 0,
    "comments_disabled": 0,
    "ratings_disabled": 0,
    "video_error_or_removed": 0,
}


def safe_int(value, field_name):
    try:
        if value is None or (isinstance(value, float) and value != value):
            raise ValueError("blank")
        text = str(value).strip()
        if text == "" or text.lower() == "nan":
            raise ValueError("blank")
        return int(float(text))
    except (ValueError, TypeError):
        failed[field_name] += 1
        return 0


def safe_bool(value, field_name):
    try:
        if isinstance(value, bool):
            return value
        text = str(value).strip().lower()
        if text in ("true", "1", "yes"):
            return True
        if text in ("false", "0", "no", ""):
            return False
        raise ValueError("not a clear boolean")
    except (ValueError, TypeError):
        failed[field_name] += 1
        return False


def parse_trending_date(text):
    parts = str(text).strip().split(".")
    year = 2000 + int(parts[0])
    day = int(parts[1])
    month = int(parts[2])
    return date(year, month, day)


def parse_publish_time(text):
    text = str(text).strip().replace("Z", "+00:00")
    return datetime.fromisoformat(text)


cleaned_records = []

for row in records:
    clean = dict(row)

    clean["views"] = safe_int(row.get("views"), "views")
    clean["likes"] = safe_int(row.get("likes"), "likes")
    clean["dislikes"] = safe_int(row.get("dislikes"), "dislikes")
    clean["comment_count"] = safe_int(row.get("comment_count"), "comment_count")

    clean["comments_disabled"] = safe_bool(row.get("comments_disabled"), "comments_disabled")
    clean["ratings_disabled"] = safe_bool(row.get("ratings_disabled"), "ratings_disabled")
    clean["video_error_or_removed"] = safe_bool(
        row.get("video_error_or_removed"), "video_error_or_removed"
    )

    try:
        clean["trending_date"] = parse_trending_date(row.get("trending_date"))
    except (ValueError, TypeError, IndexError):
        failed["trending_date"] += 1
        clean["trending_date"] = None

    try:
        clean["publish_time"] = parse_publish_time(row.get("publish_time"))
    except (ValueError, TypeError):
        failed["publish_time"] += 1
        clean["publish_time"] = None

    try:
        clean["category_id"] = int(row.get("category_id"))
    except (ValueError, TypeError):
        clean["category_id"] = None

    cleaned_records.append(clean)

print("Parse failures per field:")
for field, count in failed.items():
    print(f"  {field}: {count}")
print(f"Cleaned records kept: {len(cleaned_records):,}")

Parse failures per field:
  views: 0
  likes: 12
  dislikes: 0
  comment_count: 36
  trending_date: 0
  publish_time: 0
  comments_disabled: 0
  ratings_disabled: 0
  video_error_or_removed: 0
Cleaned records kept: 46,230


##Core Data Structure

list of dicts - Easy to loop over every video row and its mutable

tuple - Snapshot of key fields that should not change by accident and its immutable

set - Unique video IDs, fast intersection, difference across countries and its mutable

dict counter - Count how often each channel appears and its mutable

slicing - Show only the top 15 after sorting and used for list operation

nested dict - country → category → no of records and dict is mutable




In [42]:
first = cleaned_records[0]
sample_key = (
    first["video_id"],
    first["trending_date"],
    first["views"],
    first["likes"],
)
print(f"Sample key tuple: {sample_key}")

ids_by_country = {}
for code in COUNTRIES:
    ids_by_country[code] = set()

for row in cleaned_records:
    ids_by_country[row["country"]].add(row["video_id"])

us_ids = ids_by_country["US"]
gb_ids = ids_by_country["GB"]
in_ids = ids_by_country["IN"]

us_and_gb = us_ids & gb_ids
multi_country = (us_ids & gb_ids) | (us_ids & in_ids) | (gb_ids & in_ids)

only_us = us_ids - gb_ids - in_ids
only_gb = gb_ids - us_ids - in_ids
only_in = in_ids - us_ids - gb_ids

print(f"Unique videos US={len(us_ids):,} | GB={len(gb_ids):,} | IN={len(in_ids):,}")
print(f"US ∩ GB (same video_id in both): {len(us_and_gb):,}")
print(f"In at least two countries: {len(multi_country):,}")
print(f"Only US={len(only_us):,} | Only GB={len(only_gb):,} | Only IN={len(only_in):,}")

channel_counts = {}
for row in cleaned_records:
    channel = row["channel_title"]
    if channel in channel_counts:
        channel_counts[channel] += 1
    else:
        channel_counts[channel] = 1

sorted_channels = sorted(channel_counts.items(), key=lambda item: item[1], reverse=True)
top_15 = sorted_channels[:15]

print("\nTop 15 channels by trending appearances:")
for rank, (name, count) in enumerate(top_15, start=1):
    print(f"  {rank:2d}. {name}: {count}")

summary = {}
for row in cleaned_records:
    country = row["country"]
    cat_id = row["category_id"]
    cat_name = category_names.get(cat_id, "Unknown")

    if country not in summary:
        summary[country] = {}
    if cat_name not in summary[country]:
        summary[country][cat_name] = 0
    summary[country][cat_name] += 1

print("\nSample nested summary (first 5 categories per country):")
for country in COUNTRIES:
    print(f"  {country}:")
    items = list(summary[country].items())[:5]
    for cat_name, count in items:
        print(f"    {cat_name}: {count}")

Sample key tuple: ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
Unique videos US=5,068 | GB=4,113 | IN=5,878
US ∩ GB (same video_id in both): 134
In at least two countries: 340
Only US=4,834 | Only GB=3,873 | Only IN=5,652

Top 15 channels by trending appearances:
   1. QuietChallenge75: 2464
   2. OfficialReview41: 1586
   3. UltimateUnboxing65: 1433
   4. InsaneReaction46: 1233
   5. Brand NewChallenge45: 1175
   6. CrazyTrailer91: 1018
   7. FinalTrailer17: 911
   8. UltimateSpeech20: 770
   9. Brand NewSecret51: 684
  10. WildHighlights7: 682
  11. SurprisingHighlights41: 650
  12. FinalBreakdown11: 643
  13. CrazyRecipe67: 603
  14. QuietInterview9: 570
  15. UltimateVlog96: 565

Sample nested summary (first 5 categories per country):
  US:
    People & Blogs: 1423
    News & Politics: 755
    Music: 3052
    Education: 561
    Entertainment: 4371
  GB:
    Science & Technology: 470
    Howto & Style: 588
    Comedy: 941
    Entertainment: 2844
    Music: 5013
  IN:
  

##Scope

A variable made **outside** a function is global variable and functions can read it.

A variable made **inside** a function is local variable and you cannot use it outside a function.

Trying to print the local name outside raises **NameError** because it is not available in the global scope.



In [43]:
outside_message = "I live outside the function"  # global


def scope_demo():
    inside_message = "I live only inside the function"  # local
    print(f"Inside function, outside_message = {outside_message}")
    print(f"Inside function, inside_message = {inside_message}")


scope_demo()

print(f"Outside function, outside_message = {outside_message}")

try:
    print(inside_message)
except NameError as err:
    print(f"Outside function cannot see inside_message -> {err}")

Inside function, outside_message = I live outside the function
Inside function, inside_message = I live only inside the function
Outside function, outside_message = I live outside the function
Outside function cannot see inside_message -> name 'inside_message' is not defined


##Loops over strings and sequences
Count uppercase words, !/?, and number of words in a title.


In [44]:
def title_features(title):
    if title is None:
        title = ""
    title = str(title)

    marks = 0
    for ch in title:
        if ch == "!" or ch == "?":
            marks += 1

    words = title.split()
    upper_words = 0

    for i in range(len(words)):
        word = words[i]
        letters = "".join([c for c in word if c.isalpha()])
        if letters and letters.isupper():
            upper_words += 1

    return {
        "upper_words": upper_words,
        "marks": marks,
        "word_count": len(words),
    }


sample_titles = [
    cleaned_records[0]["title"],
    "WOW! You Will Not BELIEVE This?",
    "quiet daily vlog",
]
for t in sample_titles:
    print(f"title={t!r} -> {title_features(t)}")

if "VIEW_THRESHOLD" not in globals():
    VIEW_THRESHOLD = 1_000_000

i = 0
found = None
while i < len(cleaned_records):
    row = cleaned_records[i]
    i += 1

    if row["views"] is None:
        continue

    if row["views"] > VIEW_THRESHOLD:
        found = row
        break

if found:
    print(
        f"\nFirst video above {VIEW_THRESHOLD:,} views: "
        f"{found['title'][:60]!r} ({found['views']:,} views, {found['country']})"
    )
else:
    print("\nNo video found above the view threshold.")

title='Honest Football Prank LEAKED' -> {'upper_words': 1, 'marks': 0, 'word_count': 4}
title='WOW! You Will Not BELIEVE This?' -> {'upper_words': 2, 'marks': 2, 'word_count': 6}
title='quiet daily vlog' -> {'upper_words': 0, 'marks': 0, 'word_count': 3}

First video above 1,000,000 views: 'Official Cats Speech EXCLUSIVE' (2,841,278 views, US)


##Rule Based Classification

In [45]:
if "VIRAL_VIEWS" not in globals():
    VIRAL_VIEWS = 1_000_000
if "LIKE_RATIO_HIGH" not in globals():
    LIKE_RATIO_HIGH = 0.90
if "LIKE_RATIO_LOW" not in globals():
    LIKE_RATIO_LOW = 0.50


def like_ratio(likes, dislikes):
    total = likes + dislikes
    if total == 0:
        return None
    return likes / total


def classify_video(record):
    views = record.get("views", 0) or 0
    likes = record.get("likes", 0) or 0
    dislikes = record.get("dislikes", 0) or 0
    ratings_disabled = record.get("ratings_disabled", False)

    if ratings_disabled:
        return "Unrated"

    if views == 0 or (likes + dislikes) == 0:
        return "Low Engagement"

    ratio = like_ratio(likes, dislikes)

    if views >= VIRAL_VIEWS:
        if ratio is not None and ratio >= LIKE_RATIO_HIGH:
            return "Viral Hit"
        elif ratio is not None and ratio < LIKE_RATIO_LOW:
            return "Divisive"
        else:
            return "Steady Performer"
    else:
        if ratio is not None and ratio < LIKE_RATIO_LOW and (likes + dislikes) >= 100:
            return "Divisive"
        else:
            return "Steady Performer"


demo = {
    "views": 2_500_000,
    "likes": 90_000,
    "dislikes": 5_000,
    "ratings_disabled": False,
}
print(f"Hand-trace demo label: {classify_video(demo)}")

label_counts = {}
for row in cleaned_records:
    label = classify_video(row)
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1

print("\nRecords per label:")
for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count:,}")

Hand-trace demo label: Viral Hit

Records per label:
  Steady Performer: 36,773
  Viral Hit: 8,864
  Unrated: 544
  Divisive: 49


##Refactor

**'safe_int'/'like_ratio'** - Putting common work in a function means we write it once and call it many times.


In [46]:
print(safe_int("12.0", "views"), safe_int("oops", "views"))
print(like_ratio(9, 1), like_ratio(0, 0))

def short_title(title, max_len=40):
    text = "" if title is None else str(title)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

print(short_title(cleaned_records[0]["title"]))
print(short_title(cleaned_records[1]["title"]))

12 0
0.9 None
Honest Football Prank LEAKED
Surprising Cricket Review
